## Fig. A4

In [ ]:
from netCDF4 import Dataset, MFDataset
import numpy as np
import math
from find_FW import *
from FW_analysis import *
import matplotlib.pyplot as plt
import xarray as xr 
import scipy
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
import seaborn as sns
from ozone_extremes_leap import *
import string
from matplotlib import rc
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#WACCM INT_O3_2000

nc_fid_1=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/ozone_extremes/WACCM/INT_O3_2000/CO2x1SmidEmin_yBWCN.cam.h1.0101-0300.O3.isobar.zm.runmean5d.nc')
nc_fid_z_1=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/ozone_extremes/WACCM/INT_O3_2000/CO2x1SmidEmin_yBWCN.cam.h1.0101-0300.Z3.isobar.zm.nc')
nc_fid_O3_1=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/ozone_extremes/WACCM/INT_O3_2000/CO2x1SmidEmin_yBWCN.cam.h1.0101-0300.O3.isobar.zm.nc')
nc_fid_u_1=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/ozone_extremes/WACCM/INT_O3_2000/U.101-300.zm.nc')

In [ ]:
#WACCM CLIM_O3_3D

nc_fid_2=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/WACCM/NOCHEM_2000_3D/combined/B2000WCN.NOCOUPL.e122.f19_g16.int.zm.runmean5.nc')
nc_fid_z_2=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/WACCM/NOCHEM_2000_3D/combined/B2000WCN.NOCOUPL.e122.f19_g16.int.zm.nc')
nc_fid_u_2=nc_fid_z_2
nc_fid_O3_2=nc_fid_z_2

In [ ]:
#SOCOL-MPIOM INT_O3_2000

nc_fid_3=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/ozone_extremes/SOCOL-MPIOM/INT_O3_2000/testrun_v3_v3r2_O3_u_t_swhr_2000-2199.zm.runmean5d.nc')
nc_fid_z_3=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/ozone_extremes/SOCOL-MPIOM/INT_O3_2000/testrun_v3_v3r2_z_2000-2199.zm.nc')
nc_fid_O3_3=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/ozone_extremes/SOCOL-MPIOM/INT_O3_2000/testrun_v3_v3r2_O3_u_t_swhr_2000-2199.zm.nc')
nc_fid_u_3=nc_fid_O3_3

In [ ]:
#SOCOL-MPIOM CLIM_O3_3D

nc_fid_4=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/SOCOL-MPIOM/NOCHEM_2000_3D/combined/run-3D-O3_chem_m_runmean5d.nc')
nc_fid_z_4=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/SOCOL-MPIOM/NOCHEM_2000_3D/combined/run-3D-O3_after.zm.nc')
nc_fid_u_4=xr.open_dataset('/net/hydro/chemie/mfriedel/Data/SOCOL-MPIOM/NOCHEM_2000_3D/combined/run-3D-O3_chem_m.nc')
nc_fid_O3_4=nc_fid_u_4

In [ ]:
nc_fid=[]
nc_fid_z=[]
nc_fid_O3=[]
nc_fid_u=[]

nc_fid.append(nc_fid_1)
nc_fid.append(nc_fid_2)
nc_fid.append(nc_fid_3)
nc_fid.append(nc_fid_4)


nc_fid_z.append(nc_fid_z_1)
nc_fid_z.append(nc_fid_z_2)
nc_fid_z.append(nc_fid_z_3)
nc_fid_z.append(nc_fid_z_4)


nc_fid_O3.append(nc_fid_O3_1)
nc_fid_O3.append(nc_fid_O3_2)
nc_fid_O3.append(nc_fid_O3_3)
nc_fid_O3.append(nc_fid_O3_4)


nc_fid_u.append(nc_fid_u_1)
nc_fid_u.append(nc_fid_u_2)
nc_fid_u.append(nc_fid_u_3)
nc_fid_u.append(nc_fid_u_4)

In [ ]:
%reload_ext autoreload
%autoreload 2

extreme_years=50
years = 200
lev='plev'

AO=[]

for i in range(4):
    
    if i in [0,1]:
        model='WACCM'
        z='Z3'
    
    if i in [2,3]:
        model='SOCOL'
        z='geopoth'

    nc_fid_in=nc_fid_z[i]
    
    var=nc_fid_in[z]
    plev=nc_fid_in[lev]
    lats=nc_fid_in['lat']
    time=nc_fid_in['time']
    
    if lats[0]>0:
        lats_orientation = 'negative'
    else:
        lats_orientation = 'positive'

    if plev[len(plev)-1] > 1000:
        var=var.interp(plev=100000)
        
    if plev[len(plev)-1] <= 1000 and model!='MERRA':   
        var=var.interp(plev=1000)
        
    var=var.interp(lat=np.linspace(-90,90,73))
    lats=lats.interp(lat=np.linspace(-90,90,73))
    
    var_anomalies = var.groupby("time.dayofyear") - var.groupby("time.dayofyear").mean("time") #calculate anomalies
    var_anomalies=var_anomalies.sel(lat=slice(20,90)) 
    
    lats=lats.sel(lat=slice(20,90)) #select latitudes north of 20°
    lats=np.array(lats)

    #calculate latitudinal weights
    coslat = np.cos(np.deg2rad(lats).clip(0., 1.))
    wgts = np.sqrt(coslat)[..., np.newaxis]

    gh_layer = np.array(var_anomalies)
 
    if len(np.shape(gh_layer)) == 3:
        gh_layer = np.reshape(gh_layer, (np.shape(gh_layer)[0], len(lats)))

    solver = Eof(gh_layer, weights=wgts, center=True)
        
    #calculate AO index time series based on first PC time series
    AO_x=np.reshape(solver.pcs(npcs=1, pcscaling=1), (np.shape(gh_layer)[0],))
    AO_xr = xr.DataArray(AO_x, coords=[time], dims=['time'])
    
    nof_lats = len(lats)
  
    max_AO = AO_xr.argmax()
    
    #check if AO has the correct sign
    if lats_orientation=='negative':
               
             if np.nanmean(gh_layer[max_AO,0:int(nof_lats/2)-1])-np.nanmean(gh_layer[max_AO,int(nof_lats/2)+1:nof_lats-1])<0:
                     AO_xr[:]=-AO_xr[:]
               
    else:

               if np.nanmean(gh_layer[max_AO,0:int(nof_lats/2)-3])-np.nanmean(gh_layer[max_AO,int(nof_lats/2)+3:nof_lats-1])<0:
                       AO_xr[:]=-AO_xr[:]
                  
    AO.append(AO_xr)

In [ ]:
FW_dates_all=[]
AO_FW_high_all=[]
AO_FW_low_all=[]
FW_dates_high_all=[]
FW_dates_low_all=[]
O3_lowest_indices=[]
O3_highest_indices=[]

for i in range(4):
    
    if i in [0,1]:
        O3='O3'
        z='Z3'
        u='U'
        
    if i in [2,3]:
        O3='O3_m'
        z='geopoth'
        u='u_m'
        
    nc_fid_in=nc_fid[i]
    nc_fid_z_in=nc_fid_z[i]
    nc_fid_u_in=nc_fid_u[i]
    AO_x=AO[i]
    
    O3_highest, O3_lowest, O3_lowest_date, O3_highest_date, O3_lowest_index, O3_highest_index, O3_intersect, O3_years = find_ozone_extremes(nc_fid_in, O3,  lev, years,extreme_years,model)
    FW_dates=find_FW_new_leap(nc_fid_u_in, years, u, lev, model, 50)
    
    FW_ozone_max_index = find_SSW_FW_dates(FW_dates, O3_highest)
    FW_ozone_min_index = find_SSW_FW_dates(FW_dates, O3_lowest)
    
    O3_highest_years=O3_years[O3_highest]
    O3_lowest_years=O3_years[O3_lowest]

    AO_FW_high=AO_x.sel(time=AO_x.time.dt.year.isin(O3_highest_years))
    AO_FW_high=AO_FW_high.sel(time=AO_FW_high.time.dt.month.isin([3,4,5]))
    AO_FW_low=AO_x.sel(time=AO_x.time.dt.year.isin(O3_lowest_years))
    AO_FW_low=AO_FW_low.sel(time=AO_FW_low.time.dt.month.isin([3,4,5]))
    
    AO_FW_high_all.append(AO_FW_high)
    AO_FW_low_all.append(AO_FW_low)
    FW_dates_high_all.append(FW_ozone_max_index)
    FW_dates_low_all.append(FW_ozone_min_index)
    
    FW_dates_all.append(FW_dates)
    
    O3_lowest_indices.append(O3_lowest_index)
    O3_highest_indices.append(O3_highest_index)

In [ ]:
rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
rc('text', usetex=True)

sns.set_style("ticks")

cm = 1/2.54
fig, ax = plt.subplots(2,2,figsize=(16.5*cm,8*cm), constrained_layout=True, edgecolor='k', sharex=True, sharey=True)

for i in range(4):

    if i==0: 
        axis=ax[0,0]

    if i==1: 
        axis=ax[1,0]

    if i==2: 
        axis=ax[0,1]

    if i==3: 
        axis=ax[1,1]
    
    AO_highest=AO_FW_high_all[i]
    AO_lowest=AO_FW_low_all[i]
    
    AO_highest_mean=AO_highest.groupby("time.dayofyear").mean()
    AO_lowest_mean=AO_lowest.groupby("time.dayofyear").mean()
    
    AO_highest_std=AO_highest.groupby("time.dayofyear").std()
    AO_lowest_std=AO_lowest.groupby("time.dayofyear").std()
    
    O3_highest_index=O3_highest_indces[i]
    O3_lowest_index=O3_lowest_indices[i]
    
    FW_ozone_max_index=FW_dates_high_all[i]
    FW_ozone_min_index=FW_dates_low_all[i]
    
    t_array_high, p_array_high = scipy.stats.ttest_1samp(np.reshape(np.array(AO_highest), (50,92)),0, axis=0)
    t_array_low, p_array_low = scipy.stats.ttest_1samp(np.reshape(np.array(AO_lowest), (50,92)),0, axis=0)
        
    axis.fill_between(range(len(AO_highest_mean.dayofyear)),AO_highest_mean-AO_highest_std,AO_highest_mean+AO_highest_std, color='indianred', alpha=0.1) 
    axis.fill_between(range(len(AO_lowest_mean.dayofyear)), AO_lowest_mean-AO_lowest_std,AO_lowest_mean+AO_lowest_std, color='royalblue', alpha=0.1)
                      
  #  AO_highest=np.mean(AO_highest, axis=0)    
  #  AO_lowest=np.mean(AO_lowest, axis=0)    
        
    for i in range(92):
        if p_array_high[i]<0.05:
            axis.plot(i, AO_highest_mean[i], color='firebrick',marker='.', alpha=0.5, markersize=4)
            
    for i in range(92):
        if p_array_low[i]<0.05:
            axis.plot(i, AO_lowest_mean[i], color='darkblue',marker='.',alpha=0.5, markersize=4)
  
    axis.plot(range(len(AO_highest_mean)), AO_lowest_mean,  color='darkblue', alpha=0.6, linewidth=0.7)
    axis.plot(range(len(AO_highest_mean)), AO_highest_mean, color='firebrick', alpha=0.6, linewidth=0.7)
    
    axis.axhline(y=0, color='k', linestyle='--', linewidth=0.3)  

    if i in [2,3]: shift=60+1
    else: shift=59+1
    
    axis.plot(int(np.nanmean(FW_ozone_max_index)-shift), AO_highest_mean[int(np.round(np.nanmean(FW_ozone_max_index)-shift))], marker='o', color='none', markersize=5,markeredgewidth=0.7, markeredgecolor='firebrick')
    axis.text(int(np.nanmean(FW_ozone_max_index)-shift), AO_highest_mean[int(np.round(np.nanmean(FW_ozone_max_index)-shift))]-0.45,'FSW',horizontalalignment='center', fontsize=8, color='k')
    
    axis.plot(int(np.nanmean(O3_highest_index))+1, AO_highest_mean[int(np.round(np.nanmean(O3_highest_index)))], marker='o', color='none', markersize=5,markeredgewidth=0.7, markeredgecolor='firebrick')
    axis.text(int(np.nanmean(O3_highest_index))+1, AO_highest_mean[int(np.round(np.nanmean(O3_highest_index)))]-0.45,'O3',horizontalalignment='center', fontsize=8, color='k')
             
    axis.plot(int(np.nanmean(FW_ozone_min_index)-shift), AO_lowest_mean[int(np.round(np.nanmean(FW_ozone_min_index)-shift))], marker='o', color='none', markersize=5,markeredgewidth=0.7, markeredgecolor='darkblue')
    axis.text(int(np.nanmean(FW_ozone_min_index)-shift), AO_lowest_mean[int(np.round(np.nanmean(FW_ozone_min_index)-shift))]+0.2,'FSW',horizontalalignment='center', fontsize=8, color='k')  
    
    axis.plot(int(np.nanmean(O3_lowest_index)+1), AO_lowest_mean[int(np.round(np.nanmean(O3_lowest_index)))], marker='o', color='none', markersize=5,markeredgewidth=0.7, markeredgecolor='darkblue')
    axis.text(int(np.nanmean(O3_lowest_index))+1, AO_lowest_mean[int(np.round(np.nanmean(O3_lowest_index)))]+0.2,'O3',horizontalalignment='center', fontsize=8, color='k')
    
ax[1,0].set_xticks([0,31,61,92])
ax[1,0].set_xticklabels(('Mar','Apr','May','June'), fontsize=8)
ax[1,0].set_yticks([-1.5,-1,-0.5,0,0.5,1,1.5])
ax[1,0].set_yticklabels(("-1.5","-1.0","-0.5","0.0","0.5","1.0","1.5"), fontsize=8)

ax[1,1].set_xticks([0,31,61,92])
ax[1,1].set_xticklabels(('Mar','Apr','May','June'), fontsize=8)
ax[0,0].set_yticks([-1.5,-1,-0.5,0,0.5,1,1.5])
ax[0,0].set_yticklabels(("-1.5","-1.0","-0.5","0.0","0.5","1.0","1.5"), fontsize=8)

ax[0,0].set_title("WACCM", fontsize=8, fontweight="bold")
ax[0,1].set_title("SOCOL-MPIOM", fontsize=8, fontweight="bold")

ax[0,0].text(-25,0,'INT-3D', horizontalalignment='center',fontsize=8,color='black',fontweight='bold',rotation='vertical', verticalalignment='center')
ax[1,0].text(-25,0,'CLIM-3D', horizontalalignment='center',fontsize=8,color='black',fontweight='bold',rotation='vertical', verticalalignment='center')

ax[0,0].set_ylabel("AO Index", fontsize=8)
ax[1,0].set_ylabel("AO Index", fontsize=8)

ax = ax.flat
i=0
for n, axis in enumerate(ax):
    if n in [0,1,2,3,4,6,7,8,9,10]:
        axis.text(0.02,0.90, "("+string.ascii_lowercase[n]+")", transform=axis.transAxes, size=8, weight='bold')
        i=i+1
        axis.xaxis.set_tick_params(width=0.5)
        axis.yaxis.set_tick_params(width=0.5)   
        for _,s in axis.spines.items():
            s.set_linewidth(0.5)

plt.savefig('Plots/AO_evolution.pdf')        